## Load useful libraries

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

In [2]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import FloatType

## User settings

In [3]:
output_directory = 'output'
sampling_rate = 22050
hop_length = 512 * 300
spark_memory = '70G'
percentile_cutoff = 0.8
approx_quantile_precision = 0.05

#range_min = 75.
#range_max = 60. * 10.
irq_min = 45.
irq_max = 60. * 11.

# location of tracks to analyze for building the track feature database
path_library_parquet = '../database/music/output/playlist.parquet'

## Initialize Spark session

In [4]:
conf = (
    SparkConf()
    .setAppName('AnalyzedTrackLibrary')
    .set('spark.executor.memory', spark_memory)
    .set('spark.driver.memory', spark_memory)
    .set('spark.driver.maxResultSize', spark_memory)
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

26/03/22 09:56:02 WARN Utils: Your hostname, emily-MS-7B96 resolves to a loopback address: 127.0.1.1; using 192.168.1.99 instead (on interface eno1)
26/03/22 09:56:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 09:56:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load show and track library data

In [5]:
path_show_output = output_directory + '/show_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_show = spark.read.parquet(path_show_output)

In [6]:
path_library_output = output_directory + '/library_vectors_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
sdf_library = spark.read.parquet(path_library_output)

## Join the show and track library dataframes

In [7]:
sdf_cross_joined = (
    sdf_show
    .crossJoin(sdf_library)
    .orderBy('time_step', 'id')
)

In [8]:
sdf_cross_joined.show(3)

+---------+--------------------+--------------------+---+
|time_step|          array_show|       array_library| id|
+---------+--------------------+--------------------+---+
|        0|[0.29906621575355...|[0.32796314358711...| 28|
|        0|[0.29906621575355...|[0.33253091573715...| 28|
|        0|[0.29906621575355...|[0.32921785116195...| 28|
+---------+--------------------+--------------------+---+
only showing top 3 rows



## Define a function for computing cosine similarity

In [9]:
@F.udf(returnType=FloatType())
def compute_cosine_similarity(vector1, vector2):
    cosine_dist = cosine(np.array(vector1), np.array(vector2))
    similarity_score = 1 - cosine_dist
    return float(similarity_score)

## Compute cosine similarity

In [10]:
sdf_cross_joined = (
    sdf_cross_joined
    .withColumn('cosine_similarity', compute_cosine_similarity(F.col('array_show'), F.col('array_library'))).cache()
    .drop('array_show', 'array_library')
    .orderBy('time_step', 'id')
)

In [11]:
sdf_cross_joined.show(5)

26/03/22 09:56:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+---------+---+-----------------+
|time_step| id|cosine_similarity|
+---------+---+-----------------+
|        0| 28|        0.7578512|
|        0| 28|        0.7598425|
|        0| 28|        0.7616274|
|        0| 28|        0.7632483|
|        0| 28|        0.7647267|
+---------+---+-----------------+
only showing top 5 rows



## Reduce dataset size by percentile cutoff

In [12]:
sdf_cross_joined.repartition(100)  # I just made this number up, there is no specific rationale for it.

DataFrame[time_step: bigint, id: bigint, cosine_similarity: float]

In [13]:
sdf_cross_joined.count()

19839368

In [14]:
similarity_quantile_cutoff = sdf_cross_joined.approxQuantile('cosine_similarity', [percentile_cutoff], approx_quantile_precision)

In [15]:
similarity_quantile_cutoff

[0.8752384781837463]

In [16]:
sdf_cross_joined = (
    sdf_cross_joined
    .where(F.col('cosine_similarity') >= F.lit(similarity_quantile_cutoff[0]))
    .orderBy('time_step', 'id')
)

In [17]:
sdf_cross_joined.count()

4254936

## Aggregate by (timestamp, song_id)

We retain the maximum cosine similarity per (timestamp / song ID) pair:

In [18]:
sdf_cross_joined.repartition('time_step', 'id')

sdf_agg = (
    sdf_cross_joined
    .groupBy('time_step', 'id')
    .agg(
        F.max('cosine_similarity').alias('cosine_similarity'),
    )
    .orderBy('time_step', F.desc('cosine_similarity'))
)

In [19]:
sdf_agg.show(5)

+---------+----+-----------------+
|time_step|  id|cosine_similarity|
+---------+----+-----------------+
|        0|1176|       0.97957927|
|        0|5241|       0.93918365|
|        0|4375|        0.9384119|
|        0|2173|       0.93725127|
|        0|1280|        0.9325192|
+---------+----+-----------------+
only showing top 5 rows



In [20]:
#path_agg_output = output_directory + '/agg_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
#sdf_library.write.mode('overwrite').parquet(path_agg_output)

## Record the rank per timestamp¶

In [21]:
sdf_agg.repartition('time_step')

DataFrame[time_step: bigint, id: bigint, cosine_similarity: float]

In [22]:
window_spec = Window.partitionBy('time_step').orderBy(F.desc('cosine_similarity'))

sdf_agg_ranked = (
    sdf_agg
    .orderBy(F.asc('time_step'), F.desc('cosine_similarity'))
    .withColumn('rank', F.row_number().over(window_spec))
)

In [23]:
sdf_agg_ranked.show(5)

+---------+----+-----------------+----+
|time_step|  id|cosine_similarity|rank|
+---------+----+-----------------+----+
|        0|1176|       0.97957927|   1|
|        0|5241|       0.93918365|   2|
|        0|4375|        0.9384119|   3|
|        0|2173|       0.93725127|   4|
|        0|1280|        0.9325192|   5|
+---------+----+-----------------+----+
only showing top 5 rows



## Keep only the top-ranked rows per time step

In [24]:
sdf_agg_ranked = (
    sdf_agg_ranked
    .where(F.col('rank') <= 1)
    .drop('rank', 'cosine_similarity')
    .orderBy('time_step')
)

In [25]:
sdf_agg_ranked.show(5)

+---------+----+
|time_step|  id|
+---------+----+
|        0|1176|
|        1|1176|
|        2|1176|
|        3|1176|
|        4|1176|
+---------+----+
only showing top 5 rows



In [26]:
#path_rank_output = output_directory + '/rank_spark_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.parquet'
#sdf_agg_ranked.write.mode('overwrite').parquet(path_rank_output)

In [27]:
#sdf_ids = spark.read.parquet(path_rank_output)

## Compute timestamp in seconds

In [28]:
sdf_ids = (
    sdf_agg_ranked
    .withColumn('timestamp', F.col('time_step') * (hop_length / sampling_rate))
    .drop('time_step')
)

In [29]:
sdf_ids.show(5)

+----+------------------+
|  id|         timestamp|
+----+------------------+
|1176|               0.0|
|1176| 6.965986394557823|
|1176|13.931972789115646|
|1176|20.897959183673468|
|1176| 27.86394557823129|
+----+------------------+
only showing top 5 rows



## Calculate (rough) song intervals

In [30]:
sdf_ids.repartition('id')

DataFrame[id: bigint, timestamp: double]

In [31]:
sdf_ids_agg = (
    sdf_ids
    .groupBy('id')
    .agg(
        F.min('timestamp').alias('p0'),
        F.percentile_approx('timestamp', 0.25).alias('p25'),
        F.percentile_approx('timestamp', 0.75).alias('p75'),
        F.max('timestamp').alias('p100'),
    )
    .orderBy('p0')
)

In [32]:
sdf_ids_agg.show(10)

+----+------------------+------------------+------------------+------------------+
|  id|                p0|               p25|               p75|              p100|
+----+------------------+------------------+------------------+------------------+
|1176|               0.0|20.897959183673468| 76.62585034013605| 97.52380952380952|
|3785|104.48979591836735|139.31972789115645| 215.9455782312925| 5691.210884353742|
|2716|236.84353741496597| 6276.353741496599| 6401.741496598639|  6464.43537414966|
|3865| 243.8095238095238|292.57142857142856|404.02721088435374| 459.7551020408163|
|3782|466.72108843537416| 2758.530612244898|2856.0544217687075|14147.918367346938|
| 506| 508.5170068027211| 536.3809523809524| 606.0408163265306|13207.510204081633|
|1662| 626.9387755102041| 633.9047619047619| 647.8367346938775| 654.8027210884353|
| 628| 661.7687074829931|  668.734693877551| 682.6666666666666| 689.6326530612245|
|2307| 696.5986394557823| 717.4965986394558| 773.2244897959183| 801.0884353741496|
|130

## Load the track titles and artists

In [33]:
sdf_library_names = (
    spark
    .createDataFrame(pd.read_parquet(path_library_parquet))
    .drop('path')
    .orderBy('id')
)

In [34]:
sdf_library_names.show(5)

+---+--------------------+----------------+
| id|                name|          artist|
+---+--------------------+----------------+
| 28|   On My Way to Hell|Połoz & Tinnitus|
| 32|        Militia Love|             SMP|
| 35|      Slowly Melting|       Nomeansno|
| 42|Blue Monday (Elec...|    Armada Tribe|
| 53|Undisclosed Desir...|            Muse|
+---+--------------------+----------------+
only showing top 5 rows



## Add artist/title information

In [35]:
sdf_named = (
    sdf_ids_agg.join(sdf_library_names, on = 'id', how = 'left')
)


In [36]:
sdf_named.show(10)

+----+------------------+------------------+------------------+------------------+--------------------+--------------------+
|  id|                p0|               p25|               p75|              p100|                name|              artist|
+----+------------------+------------------+------------------+------------------+--------------------+--------------------+
|3199| 975.2380952380952|  982.204081632653|1010.0680272108843|            1024.0|She's Got Little ...|john crozier harr...|
|5241|4221.3877551020405|  4256.21768707483| 6861.496598639455| 6896.326530612245|               DSM-V|              HEALTH|
|2173| 5900.190476190476| 5976.816326530612| 6137.034013605442| 6213.659863945578|  Slip Slide Melting|   For Love Not Lisa|
|4519| 8568.163265306122| 8596.027210884353| 8665.687074829932| 8693.551020408164|      Apex Predators|       P.T. Adamczyk|
|1280| 7495.401360544218| 7523.265306122449| 7578.993197278911| 7606.857142857142|Club Wedding (Jac...|Trashcan Jack, Bi...|


## Compute time ranges (IRQ and full)

In [37]:
sdf_named = (
    sdf_named
    .withColumn('range', F.col('p100') - F.col('p0'))
    .withColumn('IRQ', F.col('p75') - F.col('p25'))
    .orderBy('p25')
    .drop('id')
)

In [38]:
sdf_named.show(200)

+------------------+------------------+------------------+------------------+--------------------+--------------------+------------------+------------------+
|                p0|               p25|               p75|              p100|                name|              artist|             range|               IRQ|
+------------------+------------------+------------------+------------------+--------------------+--------------------+------------------+------------------+
|               0.0|20.897959183673468| 76.62585034013605| 97.52380952380952| Shock To The System|          Billy Idol| 97.52380952380952| 55.72789115646259|
|104.48979591836735|139.31972789115645| 215.9455782312925| 5691.210884353742|N.W.O. (Re-Record...|            Ministry| 5586.721088435374| 76.62585034013605|
| 243.8095238095238|292.57142857142856|404.02721088435374| 459.7551020408163|Marcha Funebre [E...|    Divina Blasfemia|215.94557823129253|111.45578231292518|
| 508.5170068027211| 536.3809523809524| 606.04081632

In [40]:
sdf_named_cut_min_time_diff = (
    sdf_named
    
    .where(F.col('IRQ') >= irq_min)
    .where(F.col('IRQ') < irq_max)
    
    #.where(F.col('range') >= range_min)
    #.where(F.col('range') < range_max)           
    
    .orderBy('p0')
    .drop('p100', 'IRQ')
    .withColumnRenamed('p25', 'seconds')

    .withColumn('hour', F.floor(F.col('seconds') / 3600).cast('int'))
    .withColumn('minute', F.floor((F.col('seconds') % 3600) / 60).cast('int'))
    .withColumn('sec', (F.col('seconds') % 60).cast('int'))
    .withColumn('time', F.format_string("%02d:%02d:%02d", 'hour', 'minute', 'sec'))
    .select('artist', 'name', 'time')
    .orderBy('time')
)

In [41]:
sdf_named_cut_min_time_diff.show(400)

+--------------------+--------------------+--------+
|              artist|                name|    time|
+--------------------+--------------------+--------+
|          Billy Idol| Shock To The System|00:00:20|
|            Ministry|N.W.O. (Re-Record...|00:02:19|
|    Divina Blasfemia|Marcha Funebre [E...|00:04:52|
|       Head Splitter|           Hyperpunk|00:08:56|
|       Natalia Kills|          Wonderland|00:11:57|
|            ALTIMAIT|          Underwater|00:14:58|
|            Ministry|Jesus Built My Ho...|00:18:13|
|        Phase Fatale|        Hollow Flesh|00:20:32|
|         Prg/M & Bop|     Brain Chemistry|00:22:52|
|  Alessandro Adriani|One Minute (After...|00:25:39|
|            The Cult|Coming Down (Butc...|00:29:15|
|           Rotersand|Exterminate Annih...|00:33:19|
|      A Split Second|           Crimewave|00:36:13|
|        Multiple Man|           Slow Code|00:38:25|
|         Combichrist|This S*It Will Fc...|00:40:31|
|                Muse|Compliance (Purpl...|00:

## Output Mixcloud-compatible text

In [42]:
result_path = output_directory + '/results_sr_' + str(sampling_rate) + '_hl_' + str(hop_length) + '.txt'

(
    sdf_named_cut_min_time_diff
    .withColumn('text', F.concat(F.col('artist'), F.lit(' - '), F.col('name'), F.lit(' - '), F.col('time')))
    .orderBy('time')
    .select('text')
    .write.mode("overwrite").text(result_path)    
)